In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.svm import OneClassSVM
from sklearn.metrics import precision_recall_curve, average_precision_score, classification_report
from sklearn.model_selection import train_test_split

# Load processed data

In [ ]:
df = pd.read_csv('../Data/Data_finance_lab3_featured.csv')

# Remove one-hot encoded features

In [ ]:
svm_df = df[['PRODUCT_NAME', 'LOANAMOUNT_NOT_INSURANCE','INSURANCE_FEE', 'EFF_RATE', 'COUNT_TERM_RECEIPT', 'SUM_RECEIPT_AMT', 'GENDER', 'AGE', 'City', 'ADDRESS_CURRENT_CITY', 'INCOME', 'ID', 'RESULT_YES/NO']]

In [ ]:
normal_data = svm_df[svm_df['RESULT_YES/NO'] == 1]
anomaly_data = svm_df[svm_df['RESULT_YES/NO'] == 0]
normal_train, normal_test = train_test_split(normal_data, test_size=0.2, random_state=42)
test_data = pd.concat([normal_test, anomaly_data])

test_data, val_data = train_test_split(test_data, test_size=0.5, random_state=42)

X_train = normal_train
y_train = normal_train['RESULT_YES/NO']
X_train.drop(columns=['RESULT_YES/NO'], axis=1, inplace=True)
X_test = test_data
y_test = test_data['RESULT_YES/NO']
X_test.drop(columns=['RESULT_YES/NO'], axis=1, inplace=True)
X_val = val_data
y_val = val_data['RESULT_YES/NO']
X_val.drop(columns=['RESULT_YES/NO'], axis=1, inplace=True)

# Train OneClassSVM model

In [ ]:
from sklearn.svm import OneClassSVM
from sklearn.metrics import precision_recall_curve, auc

nu_values = [
    0.01, 0.05, 0.1, 0.2, 0.5
]
gamma_values = [
    1, 5, 10, 20, 25, 30
]

best_pr_auc = 0
best_params = {}

for nu in nu_values:
    for gamma in gamma_values:
        model = OneClassSVM(kernel='rbf', nu=nu, gamma=gamma)
        model.fit(X_train)
        
        y_scores = model.decision_function(X_val)
        
        precisions, recalls, _ = precision_recall_curve(y_val, y_scores)
        pr_auc = auc(recalls, precisions)
        
        if pr_auc > best_pr_auc:
            best_pr_auc = pr_auc
            best_params = {'nu': nu, 'gamma': gamma}

print(f"Best PR AUC: {best_pr_auc}")
print(f"Best Parameters: {best_params}")


# Evaluate

In [ ]:
from sklearn.metrics import precision_recall_curve, average_precision_score, auc

import matplotlib.pyplot as plt

oc_svm = OneClassSVM(kernel='rbf', gamma=best_params.get('gamma'), nu=best_params.get('nu'))
oc_svm.fit(X_train)

In [ ]:
decision_scores = oc_svm.decision_function(X_test)  
precision, recall, thresholds = precision_recall_curve(y_test, decision_scores)

epsilon = 1e-10
f1_scores = 2 * (precision * recall) / (precision + recall + epsilon)
optimal_idx = np.argmax(f1_scores)
optimal_threshold = thresholds[optimal_idx]

auc_score = auc(recall, precision)
print(f'Precio-Recall AUC: {auc_score}')

y_pred_new = (decision_scores >= optimal_threshold).astype(int)
average_precision = average_precision_score(y_test, y_pred_new)
report = classification_report(y_test, y_pred_new)
print(report)
print(f"Average Precision: {average_precision}")


plt.figure(figsize=(8, 6))
plt.plot(recall, precision, marker='o', label='Precision-Recall Curve')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve')
plt.legend()
plt.show()